## Experiment 2 — Pima Indians Diabetes (tabular)

### Dataset
The [Pima Indians Diabetes dataset](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
contains 768 observations of female patients aged 21 or older, with 8 clinical features
and a binary outcome indicating the presence of type 2 diabetes. The dataset is moderately
imbalanced — ~35% positive (diabetes) and ~65% negative.


### Results

Hyperparameter optimization via `XGBOptClf` outperforms the default XGBoost baseline by approximately +0.04 AUC --  tuning has some meaningful impact on this dataset.

A pre-processing strategy for clinically implausible zeros (Glucose, 
BloodPressure, SkinThickness, Insulin, BMI) was evaluated:

| Strategy | Test AUC ± std  | Test MCC ± std | `\Tau` ± std |
|---|---|---|---|
| No preprocessing | 0.8344 ± 0.0303  | 0.4795 ± 0.0367 | 0.3762 ± 0.0241 |
| Zeros encoded as NaN | 0.8383 ± 0.0250 | 0.4964 ± 0.0615  | 0.3492 ± 0.0143 

Encoding zeros as NaN and delegating missing value handling to XGBoost's native sparsity-aware split finding produced the best results across all metrics. 


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import xgboost as xgb
from src.xgb_opt_clf import XGBOptClf
from src.helper_functions import *
import os
import tempfile
import optuna.visualization as vis
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import json

/Users/vkvutov/Documents/my_codes/git_projects/xgb-cat-optuna/xgb-optuna/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Study Setup

In [2]:
N_TRIALS = 100
N_FOLDS = 5
RANDOM_STATE = 42
STUDY_NAME = "xgb_opt_clf_pima"

Throughout this notebook, the development set will refer to the training set and the validation sets, while the test set will be used to report results

In [3]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
        'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=cols)
print(df.head())
print(df.shape)  # should be (768, 9)

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)


In [4]:
df["Outcome"].value_counts(normalize=True)

Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64

In [5]:
X = df.drop("Outcome", axis=1).values
y = df["Outcome"].values
columns = list(df.drop("Outcome", axis=1).columns)

In [6]:
# Helper functions 
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

def encode_zeros_as_nan(X_dev, y_dev, X_test):
    # y_dev unused — encoding is label-agnostic
    X_dev  = X_dev.copy().astype(float)
    X_test = X_test.copy().astype(float)
    for idx in [columns.index(c) for c in zero_cols]:
        X_dev[X_dev[:, idx] == 0, idx]  = np.nan
        X_test[X_test[:, idx] == 0, idx] = np.nan
    return X_dev, X_test

## I) No preprocessing applied

In [20]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger(__name__)

clf_higher_penalty = XGBOptClf(n_trials=N_TRIALS, std_penalty=1.0, nfold=N_FOLDS)
nested_cv_no_high_penalty_imp = nested_cv_score(clf_higher_penalty,
                                                X, 
                                                y,
                                                n_outer=5)

14:08:24 — Nested CV: outer fold 1/5
14:08:24 — Starting Optuna optimization for 100 trials
14:09:15 — Nested CV: fold 1 — test AUC = 0.8589, dev AUC = 0.8803, MCC = 0.4914, Brier = 0.2238, threshold = 0.3511
14:09:15 — Nested CV: outer fold 2/5
14:09:15 — Starting Optuna optimization for 100 trials
14:09:46 — Nested CV: fold 2 — test AUC = 0.8800, dev AUC = 0.9043, MCC = 0.5275, Brier = 0.1565, threshold = 0.3931
14:09:46 — Nested CV: outer fold 3/5
14:09:46 — Starting Optuna optimization for 100 trials
14:10:13 — Nested CV: fold 3 — test AUC = 0.8246, dev AUC = 0.8772, MCC = 0.4942, Brier = 0.1586, threshold = 0.3433
14:10:13 — Nested CV: outer fold 4/5
14:10:13 — Starting Optuna optimization for 100 trials
14:10:54 — Nested CV: fold 4 — test AUC = 0.8042, dev AUC = 0.9404, MCC = 0.4674, Brier = 0.1693, threshold = 0.4020
14:10:54 — Nested CV: outer fold 5/5
14:10:54 — Starting Optuna optimization for 100 trials
14:11:26 — Nested CV: fold 5 — test AUC = 0.8042, dev AUC = 0.8961, MCC 

## II) Zeros encoded as NaN 

The "encode_zeros_as_nan" function is leakage-safe ('encode_zeros_as_nan(X, y, X)' is valid); however, I am eager to illustrate the custom "nested_cv_score" function.

In [21]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(message)s",
    datefmt="%H:%M:%S"
)

clf = XGBOptClf(n_trials=N_TRIALS, std_penalty=1.0, nfold=N_FOLDS)

logger = logging.getLogger(__name__)
nested_cv_nan = nested_cv_score(
    clf, X, y,
    n_outer=5,
    preprocessor=encode_zeros_as_nan
)

14:11:26 — Nested CV: outer fold 1/5


14:11:26 — Starting Optuna optimization for 100 trials
14:11:57 — Nested CV: fold 1 — test AUC = 0.8533, dev AUC = 0.8864, MCC = 0.5524, Brier = 0.1962, threshold = 0.3565
14:11:57 — Nested CV: outer fold 2/5
14:11:57 — Starting Optuna optimization for 100 trials
14:12:44 — Nested CV: fold 2 — test AUC = 0.8698, dev AUC = 0.9163, MCC = 0.5195, Brier = 0.2104, threshold = 0.3568
14:12:44 — Nested CV: outer fold 3/5
14:12:44 — Starting Optuna optimization for 100 trials
14:13:18 — Nested CV: fold 3 — test AUC = 0.8452, dev AUC = 0.8920, MCC = 0.4819, Brier = 0.1531, threshold = 0.3211
14:13:18 — Nested CV: outer fold 4/5
14:13:18 — Starting Optuna optimization for 100 trials
14:13:59 — Nested CV: fold 4 — test AUC = 0.8260, dev AUC = 0.9140, MCC = 0.5446, Brier = 0.1684, threshold = 0.3596
14:13:59 — Nested CV: outer fold 5/5
14:13:59 — Starting Optuna optimization for 100 trials
14:14:52 — Nested CV: fold 5 — test AUC = 0.7970, dev AUC = 0.9091, MCC = 0.3837, Brier = 0.2154, threshold =

In [22]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

def baseline_nested_cv(X, y, n_outer=N_FOLDS, preprocessor=None, random_state=RANDOM_STATE):
    """Nested CV for default XGBoost — mirrors nested_cv_score structure."""
    outer_cv = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=random_state)
    scores, dev_aucs = [], []

    for fold_idx, (dev_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_dev,  y_dev  = X[dev_idx],  y[dev_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        if preprocessor is not None:
            X_dev, X_test = preprocessor(X_dev, y_dev, X_test)

        clf = XGBClassifier(random_state=random_state)
        clf.fit(X_dev, y_dev)

        dev_auc  = roc_auc_score(y_dev,  clf.predict_proba(X_dev)[:, 1])
        test_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

        scores.append(test_auc)
        dev_aucs.append(dev_auc)

    scores   = np.array(scores)
    dev_aucs = np.array(dev_aucs)
    return {"scores": scores, "dev_aucs": dev_aucs}

# Run all three combinations
baseline_no_prep = baseline_nested_cv(X, y)
baseline_nan     = baseline_nested_cv(X, y, preprocessor=encode_zeros_as_nan)

# Summary
print("=" * 60)
print("Summary — Default XGBoost")
print("=" * 60)
print(f"No preprocessing:       {baseline_no_prep['scores'].mean():.4f} ± {baseline_no_prep['scores'].std():.4f}")
print(f"Zeros as NaN:           {baseline_nan['scores'].mean():.4f} ± {baseline_nan['scores'].std():.4f}")

Summary — Default XGBoost
No preprocessing:       0.7914 ± 0.0284
Zeros as NaN:           0.7947 ± 0.0327


## Experiment Tracking with MLflow

This cell logs the full experiment to MLflow, including:

- **Parameters** — Optuna hyperparameters
- **Metrics** — CV AUC mean/std, dev/test AUC, dev/test MCC, applied threshold, nested CV scores per fold
- **Artifacts** — classification reports (JSON), ROC curve (PNG), Optuna plots (HTML), trained XGBoost model

### Nested CV
Performance is obtained via nested cross-validation (`n_outer=5`, `std_penalty=1`).
The outer loop evaluates performance; the inner loop (inside `fit()`) optimises hyperparameters via Optuna.

### Viewing Results
Launch the MLflow UI with:
```bash
cd notebooks

mlflow ui --port 5002
```
Then open `http://localhost:5002` in your browser.

In [23]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [24]:
import os, json, time, tempfile
import mlflow
import optuna.visualization as vis
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, roc_curve


mlflow.set_tracking_uri(f"file://{os.path.abspath('mlruns')}")
mlflow.set_experiment(STUDY_NAME)

with mlflow.start_run(run_name="pima_diabetes"):

    # ─── Tags 
    mlflow.set_tags({
        "model":   "XGBOptClf",
        "dataset": "pima_diabetes_no_preprocessing",
    })

    # ─── Fit 
    clf = XGBOptClf(n_trials=20, std_penalty=1.0, nfold=N_FOLDS)
    clf.fit(X_dev, y_dev)

    # ─── Params 
    mlflow.log_params({
        **clf.best_params_,
        "n_trials":    clf.n_trials,
        "n_estimators": clf.best_num_boost_round_,
        "n_jobs":      clf._resolve_n_jobs(),
        "std_penalty": clf.std_penalty,
        "eval_metric": clf.eval_metric,
        "nfold":       clf.nfold,
    })
    
    # ─── Eval Metrics 
    results = clf.eval(X_dev, y_dev, X_test, y_test)
    mlflow.log_metrics({
        "dev_auc":           results["dev_auc"],
        "dev_mcc":           results["dev_mcc"],
        "test_auc":          results["test_auc"],
        "test_mcc":          results["test_mcc"],
        "applied_threshold": results["applied_threshold"],
        "dev_brier":        results["dev_brier"],
        "test_brier":       results["test_brier"],
    })

    # ─── Nested CV 
    nested_cv = nested_cv_score(clf, X, y, n_outer=5, stratified=True, preprocessor=None)
    mlflow.log_metrics({
        "nested_cv_test_mean_auc": nested_cv["scores"].mean(),
        "nested_cv_test_std_auc":  nested_cv["scores"].std(),
        "nested_cv_dev_mean_auc":  nested_cv["dev_aucs"].mean(),
        "nested_cv_dev_std_auc":   nested_cv["dev_aucs"].std(),
        "nested_cv_test_mean_mcc":      nested_cv["test_mccs"].mean(),
        "nested_cv_test_std_mcc":       nested_cv["test_mccs"].std(),
        "nested_cv_dev_mean_mcc":      nested_cv["dev_mccs"].mean(),
        "nested_cv_dev_std_mcc":       nested_cv["dev_mccs"].std(),
        "nested_cv_mean_threshold": nested_cv["optimal_thresholds"].mean(),
        "nested_cv_std_threshold":  nested_cv["optimal_thresholds"].std(),
    })

    # ─── Nested CV per fold 
    for i, (score, mcc, threshold, n_tree, params) in enumerate(zip(
        nested_cv["scores"],
        nested_cv["test_mccs"],
        nested_cv["optimal_thresholds"],
        nested_cv["n_trees"],
        nested_cv["best_params"]
    )):
        mlflow.log_metrics({
            f"fold_{i+1}_auc":       score,
            f"fold_{i+1}_mcc":       mcc,
            f"fold_{i+1}_threshold": threshold,
            f"fold_{i+1}_n_trees":   n_tree,
        })
        mlflow.log_dict(params, f"fold_{i+1}_params.json")

    with tempfile.TemporaryDirectory() as tmp:

        # ─── Thresholds & MCC across folds 
        threshold_df = pd.DataFrame({
            "fold":      range(1, len(nested_cv["optimal_thresholds"]) + 1),
            "threshold": nested_cv["optimal_thresholds"],
            "auc":       nested_cv["scores"],
            "mcc":       nested_cv["test_mccs"],
        })
        threshold_df.loc["mean"] = ["mean",
                                     nested_cv["optimal_thresholds"].mean(),
                                     nested_cv["scores"].mean(),
                                     nested_cv["test_mccs"].mean()]
        threshold_df.loc["std"]  = ["std",
                                     nested_cv["optimal_thresholds"].std(),
                                     nested_cv["scores"].std(),
                                     nested_cv["test_mccs"].std()]
        print(threshold_df.to_string(index=False))
        path = os.path.join(tmp, "thresholds_across_folds.csv")
        threshold_df.to_csv(path, index=False)
        mlflow.log_artifact(path)

        # ─── ROC Curve 
        fpr, tpr, thresholds = roc_curve(y_test, clf.predict_proba(X_test)[:, 1])
        idx = np.argmin(np.abs(thresholds - results["applied_threshold"]))
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, label=f"AUC = {clf.score(X_test, y_test):.4f}", linewidth=2)
        ax.plot([0, 1], [0, 1], "--", color="gray")
        ax.scatter(fpr[idx], tpr[idx], color="red", zorder=5,
                label=f"Applied Threshold = {results['applied_threshold']:.2f}")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curve")
        ax.legend()
        fig.savefig(os.path.join(tmp, "roc_curve.png"))
        mlflow.log_artifact(os.path.join(tmp, "roc_curve.png"))
        plt.close(fig)

        # ─── Optuna Plots 
        fig1, fig2, fig3, fig4 = clf.plot_optuna_insights()
        for name, fig in [
            ("optimization_history", fig1),
            ("param_importances",    fig2),
            ("slice",                fig3),
            ("parallel_coordinate",  fig4),
        ]:
            path = os.path.join(tmp, f"{name}.html")
            fig.write_html(path)
            mlflow.log_artifact(path)

        # ─── Log Model ────────────────────────────────────────────────────────
        mlflow.xgboost.log_model(clf.best_model_, artifact_path="model")

14:28:28 — Starting Optuna optimization for 20 trials
14:28:39 — Nested CV: outer fold 1/5
14:28:39 — Starting Optuna optimization for 20 trials
14:28:51 — Nested CV: fold 1 — test AUC = 0.8641, dev AUC = 0.8977, MCC = 0.5462, Brier = 0.1470, threshold = 0.3524
14:28:51 — Nested CV: outer fold 2/5
14:28:51 — Starting Optuna optimization for 20 trials
14:29:04 — Nested CV: fold 2 — test AUC = 0.8806, dev AUC = 0.8955, MCC = 0.5761, Brier = 0.1333, threshold = 0.3973
14:29:04 — Nested CV: outer fold 3/5
14:29:04 — Starting Optuna optimization for 20 trials
14:29:13 — Nested CV: fold 3 — test AUC = 0.8252, dev AUC = 0.8952, MCC = 0.4260, Brier = 0.1589, threshold = 0.3089
14:29:13 — Nested CV: outer fold 4/5
14:29:13 — Starting Optuna optimization for 20 trials
14:29:22 — Nested CV: fold 4 — test AUC = 0.8109, dev AUC = 0.8890, MCC = 0.4504, Brier = 0.2231, threshold = 0.3519
14:29:22 — Nested CV: outer fold 5/5
14:29:22 — Starting Optuna optimization for 20 trials
14:29:35 — Nested CV: f

fold  threshold      auc      mcc
   1   0.352376 0.864074 0.546160
   2   0.397342 0.880556 0.576066
   3   0.308872 0.825185 0.425957
   4   0.351946 0.810943 0.450409
   5   0.368317 0.792830 0.356667
mean   0.355771 0.834718 0.471052
 std   0.028675 0.032783 0.080265


2026/04/27 14:29:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/04/27 14:29:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
